Let's take a look at the reconstructions in more detail.

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
from data_modules.mnms import MnMsDataModule
import lightning as L
import torch

from arch.unet.acunet import ACUNet
from arch.unet.unet import UNet
from utils.const import SEED
from data_modules.acdc import ACDCDataModule
from utils.utils import setup_device

model_path = "logs/unet_mnms/cross-entropy-centre-1-few/checkpoints/epoch=25-step=156.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
# data_module = ACDCDataModule(batch_size=12)
data_module = MnMsDataModule(
    batch_size=24,
    from_centre=1,
    num_subjects=5,
    sort_by_validity=True,
)

# Load model
model = UNet.load_from_checkpoint(model_path)
# model = ACUNet.load_from_checkpoint(model_path)

# Reseed after preprocessing data
L.seed_everything(SEED)

In [ ]:
loader_test = data_module.test_dataloader(shuffle=True)
x, y, _, _ = next(iter(loader_test))
print(x.shape)
print(y.shape)

In [ ]:
from utils.eval import get_samples_and_reconstructions_pixel_diff
from utils.utils import show_samples

with torch.no_grad():
    model.eval()
    model.to(device)
    x = x.to(device)

    y_hat_logits = model(x)

y_hat_logits = y_hat_logits.cpu()

samples, reconstructions, reconstruction_pixel_error = get_samples_and_reconstructions_pixel_diff(
    y,
    y_hat_logits,
    return_reconstructions=True,
)

In [ ]:
show_samples(samples, rgb=False, ncol=12, figsize=(48, 4))
show_samples(reconstructions, rgb=False, ncol=12, figsize=(48, 4))
show_samples(samples, reconstruction_pixel_error, rgb=False, ncol=12, figsize=(48, 4))

In [ ]:
from utils.utils import show_scans_and_masks

show_scans_and_masks(
    x,
    samples,
    ncol=12,
    figsize=(48, 4),
)